# 03 · U-Net

In the previous notebook we saw that a convolution layer produces feature maps, and that the deeper we go in the network, the smaller and more abstract the maps get.

That's fine for saying *what* is in the image. But to segment, we need to say *where* each thing is, pixel by pixel. So we need an output that has the same size as the input image.

U-Net solves this in two stages:
- a part that goes down (the encoder): the image shrinks and the number of channels grows, like in YOLO;
- a part that goes back up (the decoder): we enlarge again until we get back to the original size.

In between, **skip connections** copy the encoder's maps straight into the decoder. We'll build the network, train it on photos of dogs and cats, then see what happens when we remove these skip connections.

Let's start with the imports.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time

from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

## 1. The basic brick: DoubleConv

The whole U-Net is built from the same small block: two 3×3 convolutions in a row, each followed by a BatchNorm and a ReLU.

With `padding=1`, the image size doesn't change. Only the number of channels changes (here from 3 to 32). BatchNorm brings the values back to a reasonable scale, which makes training more stable. We use `bias=False` because BatchNorm already has its own offset, so the convolution's bias would be useless.

The small test at the bottom just checks that the output shape is right.

In [ ]:

class DoubleConv(nn.Module):
    """(conv 3x3 -> BatchNorm -> ReLU) x 2. Changes the channels, keeps the size."""

    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, kernel_size=3, stride=1, padding=1, bias=False)
        self.batch1 = nn.BatchNorm2d(c_out)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(c_out, c_out, kernel_size=3, stride=1, padding=1, bias=False)
        self.batch2 = nn.BatchNorm2d(c_out)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.batch1(x)
        x = self.relu1(x)

        x = self.conv2(x)
        x = self.batch2(x)
        x = self.relu2(x)

        return x


if __name__ == "__main__":
    block = DoubleConv(3, 32)
    x = torch.randn(2, 3, 128, 128)          # (batch, channels, H, W)
    y = block(x)
    print("output:", tuple(y.shape))
    assert y.shape == (2, 32, 128, 128), "the shape should be (2, 32, 128, 128)"

    n = sum(p.numel() for p in block.parameters())
    print("parameters:", n)

## 2. Going down, going up, and the full network

**Down**: a 2×2 max-pooling halves the size, then a DoubleConv increases the number of channels.

**Up**: a transposed convolution (`ConvTranspose2d`) does the opposite, it doubles the size and halves the channels. Then, if skip connections are on, we stick the encoder map that had the same size next to it (`torch.cat`). That's why the following DoubleConv receives `c_in` channels instead of `c_in // 2`.

**MiniUNet**: 4 steps down, 4 steps up. With `base=32`, the channels go 32 → 64 → 128 → 256 → 512 on the way down, then the reverse on the way up. At the end, a 1×1 convolution gives one score per class for each pixel.

Why skip connections? At the bottom of the network, the image is only 8×8. The network knows there's a cat, but it has lost the details of the outlines. Skip connections give those details back on the way up.

The `skip` parameter lets us build both versions of the network so we can compare them later.

In [ ]:
class Down(nn.Module):
    """2x2 max-pool then DoubleConv: size / 2, channels c_in -> c_out"""
    def __init__(self, c_in, c_out):
        super().__init__()
        self.max_pool = nn.MaxPool2d(kernel_size=2)
        self.doubleConv = DoubleConv(c_in, c_out)

    def forward(self, x):
        x = self.max_pool(x)
        x = self.doubleConv(x)
        return x


class Up(nn.Module):
    """2x2 up-conv (channels / 2, size x 2), concat with the skip, then DoubleConv"""
    def __init__(self, c_in, c_out, skip=True):
        super().__init__()
        self.skip = skip
        self.up = nn.ConvTranspose2d(c_in, c_in // 2, kernel_size=2, stride=2)
        channel_in = c_in if self.skip else c_in // 2
        self.conv = DoubleConv(channel_in, c_out)

    def forward(self, x, skip_connection):
        x = self.up(x)
        if self.skip:
            x = torch.cat((x, skip_connection), 1)

        x = self.conv(x)
        return x


class MiniUNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=1, base=32, skip=True):
        super().__init__()

        # Encoder: 3 -> 32 -> 64 -> 128 -> 256, bottleneck: 512
        self.inc   = DoubleConv(in_channels, base)
        self.down1 = Down(base, base * 2)
        self.down2 = Down(base * 2, base * 4)
        self.down3 = Down(base * 4, base * 8)
        self.down4 = Down(base * 8, base * 16)

        # Decoder: up1 (512 -> 256), up2, up3, up4 (64 -> 32)
        self.up1 = Up(base * 16, base * 8, skip)
        self.up2 = Up(base * 8, base * 4, skip)
        self.up3 = Up(base * 4, base * 2, skip)
        self.up4 = Up(base * 2, base, skip)

        # Output: 1x1 conv from base to n_classes
        self.outc = nn.Conv2d(base, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        return self.outc(x)


if __name__ == "__main__":
    model = MiniUNet(in_channels=3, n_classes=1)
    model_2 = MiniUNet(in_channels=3, n_classes=1, skip=False)
    x = torch.randn(2, 3, 128, 128)
    y = model(x)
    print("output:", tuple(y.shape))
    assert y.shape == (2, 1, 128, 128)

    n = sum(p.numel() for p in model.parameters())
    print(f"parameters: {n:,}")
    assert n == 7_763_041, "unexpected number of parameters"

    n = sum(p.numel() for p in model_2.parameters())
    print(f"parameters for the model without skip connections: {n:,}")
    assert n == 6_979_681, "unexpected number of parameters"

Where are the parameters? Let's look block by block. Most of them sit at the bottom of the network, where there are the most channels.

In [ ]:
# total
print(sum(p.numel() for p in model.parameters()))

# breakdown per block
for name, block in model.named_children():
    print(f"{name:6s} {sum(p.numel() for p in block.parameters()):>12,}")

## 3. The data: Oxford-IIIT Pet

It's a dataset of about 7,000 photos of dogs and cats, with a mask for each photo. In the mask, each pixel is 1 (animal), 2 (background) or 3 (border, the blurry area around the animal). We subtract 1 to get classes 0, 1, 2, which is what PyTorch expects.

Two important points in `transform_pair`:
- the image and the mask must go through exactly the same resizing, otherwise they no longer match;
- for the mask we use `NEAREST` interpolation. With bilinear interpolation, a pixel between class 0 and class 2 could become 1, which makes no sense.

We keep 500 images from `trainval` for validation. The `test` split is only used at the very end.

In [ ]:
SIZE = 128
MEAN = [0.485, 0.456, 0.406]   # ImageNet mean and standard deviation, a common standard
STD = [0.229, 0.224, 0.225]
CLASSES = ["animal", "background", "border"]


def transform_pair(image, mask):
    """Transforms the image AND its mask together."""
    # Image: bilinear interpolation (smooth)
    image = TF.resize(image, [SIZE, SIZE], interpolation=InterpolationMode.BILINEAR)
    image = TF.to_tensor(image)                 # (3, H, W), float in [0, 1]
    image = TF.normalize(image, MEAN, STD)

    # Mask: NEAREST interpolation (so we always get whole class numbers, never in-between values)
    mask = TF.resize(mask, [SIZE, SIZE], interpolation=InterpolationMode.NEAREST)
    mask = torch.as_tensor(np.array(mask), dtype=torch.long) - 1   # (H, W), values 0, 1, 2
    return image, mask


def load_data(root="../../data", batch_size=16, n_val=500, seed=42):
    trainval = datasets.OxfordIIITPet(root=root, split="trainval", target_types="segmentation", transforms=transform_pair, download=True)
    test = datasets.OxfordIIITPet(root=root, split="test", target_types="segmentation", transforms=transform_pair, download=True)

    # Split trainval into train / validation, with a fixed seed so results are reproducible
    g = torch.Generator().manual_seed(seed)
    train, val = random_split(trainval, [len(trainval) - n_val, n_val], generator=g)

    train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader

The first run downloads the dataset (about 800 MB) into `data/`.

In [ ]:
train_loader, val_loader, test_loader = load_data()

Before training anything, we look at the data. We check the shapes, the mask values, and the share of each class. The background takes more than half of the pixels: a model that answered "background" everywhere would already be right more than 50% of the time. That's why we won't rely on accuracy alone.

In [ ]:
print("train:", len(train_loader.dataset), " val:", len(val_loader.dataset), " test:", len(test_loader.dataset))

# a) Shapes, types and values
images, masks = next(iter(train_loader))
print("images:", tuple(images.shape), images.dtype)
print("masks:", tuple(masks.shape), masks.dtype)
print("mask values:", torch.unique(masks).tolist())

# b) Share of each class (useful for the loss)
for c, name in enumerate(CLASSES):
    print(f"  {name:10s} {(masks == c).float().mean():.1%}")

# c) Look at the images with their masks
def denormalize(img):
    return (img * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k in range(4):
    axes[0, k].imshow(denormalize(images[k]).permute(1, 2, 0))   # (C,H,W) -> (H,W,C) for matplotlib
    axes[1, k].imshow(masks[k], vmin=0, vmax=2)
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.tight_layout(); plt.show()

## 4. Overfitting test

A useful habit before a long training run: take a single batch of 8 images and ask the network to learn it by heart.

If everything is wired correctly, the loss should drop almost to zero. If it doesn't move, there's a bug somewhere (wrong shape, wrong loss, shifted mask…), and there's no point launching 15 epochs.

We use `CrossEntropyLoss` because we have 3 classes: for each pixel, the network outputs 3 scores and the loss compares them with the right class.

In [ ]:
# Overfitting test

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# A single batch, always the same one
train_loader, _, _ = load_data(batch_size=8)
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)

model = MiniUNet(in_channels=3, n_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for it in range(201):
    logits = model(images)
    loss = criterion(logits, masks)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if it % 20 == 0:
        with torch.no_grad():
            pred = logits.argmax(dim=1)                 # (8, 128, 128): predicted class per pixel
            acc = (pred == masks).float().mean()
        print(f"it {it:3d}   loss {loss.item():.4f}   pixel accuracy {acc:.1%}")

Same test, without skip connections. Compare how fast the loss goes down.

In [ ]:
# Overfitting test

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# A single batch, always the same one
train_loader, _, _ = load_data(batch_size=8)
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)

model = MiniUNet(in_channels=3, n_classes=3, skip=False).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for it in range(201):
    logits = model(images)
    loss = criterion(logits, masks)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if it % 20 == 0:
        with torch.no_grad():
            pred = logits.argmax(dim=1)                 # (8, 128, 128): predicted class per pixel
            acc = (pred == masks).float().mean()
        print(f"it {it:3d}   loss {loss.item():.4f}   pixel accuracy {acc:.1%}")

## 5. Full training

`train_epoch` does one pass over all the training images. `evaluate` computes the validation loss and, more importantly, the **IoU** of each class.

IoU (*Intersection over Union*) measures how well the predicted mask covers the true mask: we divide the pixels where both agree by all the pixels where at least one of them sees the class. 1 means perfect, 0 means missed. **mIoU** is the average over the 3 classes. Unlike accuracy, it isn't fooled by the background taking up all the space.

At each epoch, if the mIoU beats the previous best, we save the model in `models/`.

Warning: 15 epochs take time, especially without a GPU. Already-trained models are in `models/`, so you can jump straight to part 6.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        loss = criterion(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * images.size(0)
        n += images.size(0)
    return total / n


@torch.no_grad()
def evaluate(model, loader, criterion, device, n_classes=3):
    model.eval()
    total, n = 0.0, 0
    intersection = torch.zeros(n_classes)
    union = torch.zeros(n_classes)
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        total += criterion(logits, masks).item() * images.size(0)
        n += images.size(0)
        pred = logits.argmax(dim=1)

        for c in range(n_classes):
            mask_pred = pred == c
            mask_true = masks == c

            intersection[c] += (mask_true & mask_pred).sum().item()
            union[c] += (mask_pred | mask_true).sum().item()

    iou = intersection / union.clamp(min=1)
    return total / n, iou


if __name__ == "__main__":

    # Training on the dataset with skip connections

    torch.manual_seed(0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)
    N_EPOCHS = 15

    train_loader, val_loader, _ = load_data(batch_size=16)   # the test set is only used at the very end
    model = MiniUNet(in_channels=3, n_classes=3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best = 0.0
    for ep in range(1, N_EPOCHS + 1):
        t0 = time.time()
        loss_train = train_epoch(model, train_loader, criterion, optimizer, device)
        loss_val, iou = evaluate(model, val_loader, criterion, device)
        miou = iou.mean().item()

        details = "  ".join(f"{name} {v:.3f}" for name, v in zip(CLASSES, iou.tolist()))
        print(f"ep {ep:2d} | train {loss_train:.4f} | val {loss_val:.4f} | {details} | mIoU {miou:.3f} | {time.time() - t0:.0f}s")

        if miou > best:
            best = miou
            torch.save(model.state_dict(), "../../models/unet_best.pt")
            print("        -> best model saved")

Same training, this time without skip connections.

In [ ]:
# Training without skip connections

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
N_EPOCHS = 15

train_loader, val_loader, _ = load_data(batch_size=16)   # the test set is only used at the very end
model = MiniUNet(in_channels=3, n_classes=3, skip=False).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best = 0.0
for ep in range(1, N_EPOCHS + 1):
    t0 = time.time()
    loss_train = train_epoch(model, train_loader, criterion, optimizer, device)
    loss_val, iou = evaluate(model, val_loader, criterion, device)
    miou = iou.mean().item()

    details = "  ".join(f"{name} {v:.3f}" for name, v in zip(CLASSES, iou.tolist()))
    print(f"ep {ep:2d} | train {loss_train:.4f} | val {loss_val:.4f} | {details} | mIoU {miou:.3f} | {time.time() - t0:.0f}s")

    if miou > best:
        best = miou
        torch.save(model.state_dict(), "../../models/unet_best_sans_skip.pt")
        print("        -> best model saved")

## 6. Results

We reload the best model and display 6 validation images: the photo, the true mask, then the predicted mask.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MiniUNet(in_channels=3, n_classes=3).to(device)
model.load_state_dict(torch.load("../../models/unet_best.pt", map_location=device))
model.eval()

_, val_loader, _ = load_data(batch_size=8)
images, masks = next(iter(val_loader))
with torch.no_grad():
    pred = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for k in range(6):
    axes[0, k].imshow(denormalize(images[k]).permute(1, 2, 0))
    axes[1, k].imshow(masks[k], vmin=0, vmax=2)
    axes[2, k].imshow(pred[k], vmin=0, vmax=2)
    for row in range(3):
        axes[row, k].axis("off")
axes[0, 0].set_title("image", loc="left")
axes[1, 0].set_title("ground truth", loc="left")
axes[2, 0].set_title("prediction", loc="left")
plt.tight_layout(); plt.show()

Same for the model without skip connections. Look mostly at the animals' outlines and at the "border" class.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MiniUNet(in_channels=3, n_classes=3, skip=False).to(device)
model.load_state_dict(torch.load("../../models/unet_best_sans_skip.pt", map_location=device))
model.eval()

_, val_loader, _ = load_data(batch_size=8)
images, masks = next(iter(val_loader))
with torch.no_grad():
    pred = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for k in range(6):
    axes[0, k].imshow(denormalize(images[k]).permute(1, 2, 0))
    axes[1, k].imshow(masks[k], vmin=0, vmax=2)
    axes[2, k].imshow(pred[k], vmin=0, vmax=2)
    for row in range(3):
        axes[row, k].axis("off")
axes[0, 0].set_title("image", loc="left")
axes[1, 0].set_title("ground truth", loc="left")
axes[2, 0].set_title("prediction", loc="left")
plt.tight_layout(); plt.show()

We redefine `evaluate` here so the last cell can run without re-running the training.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, n_classes=3):
    model.eval()
    total, n = 0.0, 0
    intersection = torch.zeros(n_classes)
    union = torch.zeros(n_classes)
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        total += criterion(logits, masks).item() * images.size(0)
        n += images.size(0)
        pred = logits.argmax(dim=1)

        for c in range(n_classes):
            mask_pred = pred == c
            mask_true = masks == c

            intersection[c] += (mask_true & mask_pred).sum().item()
            union[c] += (mask_pred | mask_true).sum().item()

    iou = intersection / union.clamp(min=1)
    return total / n, iou

## 7. Final comparison on the test set

Both models are evaluated on the test images, which they never saw during training.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

_, _, test_loader = load_data(batch_size=16)
criterion = nn.CrossEntropyLoss()

for name, file, skip in [("with skips", "../../models/unet_best.pt", True),
                         ("without skips", "../../models/unet_best_sans_skip.pt", False)]:
    model = MiniUNet(in_channels=3, n_classes=3, skip=skip).to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    loss, iou = evaluate(model, test_loader, criterion, device)
    details = "  ".join(f"{c} {v:.3f}" for c, v in zip(CLASSES, iou.tolist()))
    print(f"{name:13s} | {details} | mIoU {iou.mean():.3f}")

## Key takeaways

- U-Net goes down to understand the image, then back up to recover the position of each pixel.
- Skip connections bring back the details lost on the way down. Without them, the network roughly knows where the animal is, but the outlines are blurry.
- Before a long training run, check that the network can learn a single batch by heart.
- For segmentation, we measure with IoU, not with accuracy.

## Exercises

1. In the results, which class has the worst IoU? Why do you think so?
2. Change `base=32` to `base=16`. How many parameters are left? Is the result much worse?
3. Remove only the top skip connection (`up4`) and keep the others. What changes?